In [1]:
import Pkg
Pkg.add([
    "IndividualDisplacements",
    "NCDatasets",
    "Interpolations",
    "DataFrames",
    "Dates",
    "GeoMakie",      # for Arctic map plots
    "CairoMakie",
])

   Resolving package versions...
  No Changes to `~/OceanBiome1D/Project.toml`
  No Changes to `~/OceanBiome1D/Manifest.toml`
Precompiling packages...
           ✗ IndividualDisplacements
  0 dependencies successfully precompiled in 26 seconds. 512 already precompiled.
  1 dependency errored.
  For a report of the errors see `julia> err`. To retry use `pkg> precompile`


In [ ]:
using NCDatasets, Dates

ds = Dataset("/work/scratch-pw5/thopri/cmems/cmems_mod_arc_phy_my_topaz4_P1M_vyo-vxo_180.00W-179.88E_50.00N-90.00N_0.00-4000.00m_1991-01-01-2026-01-01.nc")

lon  = Array(ds["longitude"])   # (nx,)
lat  = Array(ds["latitude"])    # (ny,)
time = Array(ds["time"])        # (nt,)  — usually DateTime or Dates.DateTime

# velocity components — shape (nx, ny, nt) or (nx, ny, nz, nt) if 3-D
u_raw = Array(ds["vxo"])         # eastward velocity  [m/s]
v_raw = Array(ds["vyo"])         # northward velocity [m/s]

close(ds)

# Take surface layer if 3-D
u = u_raw[:, 1, :, :]   # (nx, ny, nt)
v = v_raw[:, 1, :, :]

# Replace fill values with 0 (land) — CMEMS fill is typically 9.96921e36
fill_val = 9.96921f36
u[abs.(u) .> 1e10] .= 0f0
v[abs.(v) .> 1e10] .= 0f0

In [ ]:
using Interpolations

# Convert time to Float64 (seconds since epoch) for the interpolator
t_sec = Float64.(Dates.value.(time .- time[1])) ./ 1e3  # milliseconds → seconds

# Build interpolation objects: (lon, lat, time) → u or v
# BSpline(Linear()) + Flat() boundary is safe for ocean edges
itp_u = interpolate((lon, lat, t_sec), u, Gridded(Linear()))
itp_v = interpolate((lon, lat, t_sec), v, Gridded(Linear()))

# Extrapolation: clamp to boundary (flat) avoids blow-up outside the domain
eu = extrapolate(itp_u, Flat())
ev = extrapolate(itp_v, Flat())

In [ ]:
# (river name, lon, lat)
river_mouths = [
    ("Ob",        73.0,  68.5),
    ("Yenisei",   82.0,  71.8),
    ("Lena",     126.5,  72.4),
    ("Kolyma",   161.0,  68.7),
    ("Mackenzie",-134.0, 69.4),
    ("Pechora",   57.0,  68.2),
]

# Scatter multiple particles around each mouth (e.g. 3×3 grid, 0.5° apart)
function seed_around(lon0, lat0; n=3, δ=0.5)
    offsets = range(-δ*(n÷2), step=δ, length=n)
    [(lon0+dx, lat0+dy) for dx in offsets, dy in offsets] |> vec
end

seed_points = vcat([seed_around(lo, la) for (_, lo, la) in river_mouths]...)
# → Vector of (lon, lat) tuples, one per particle

In [ ]:
using IndividualDisplacements, DataFrames

# 1. Wrap your interpolators in a FlowFields-compatible function.
#    The package expects a function f!(du, u, p, t) in the ODE sense,
#    where u = [lon, lat] and p is any parameter struct.

function velocity!(du, pos, p, t)
    lon_p, lat_p = pos[1], pos[2]
    du[1] = eu(lon_p, lat_p, t) / (111_320.0 * cosd(lat_p))  # m/s → °/s
    du[2] = ev(lon_p, lat_p, t) / 111_320.0                   # m/s → °/s
end

# 2. Initial conditions — one column per particle
n_part   = length(seed_points)
x0       = [p[1] for p in seed_points]
y0       = [p[2] for p in seed_points]

# IndividualDisplacements uses a DataFrame for state
I = Individuals(
    📌 = [x0 y0]',      # 2×n_part matrix of [lon; lat]
    🔴 = velocity!,     # the velocity function
    🌐 = DataFrame(),   # optional output collector
)

# 3. Integration period
t_start = 0.0
t_end   = 90 * 86400.0     # 90 days in seconds

∫!(I, (t_start, t_end))    # runs all particles forward

# Result: I.🌐 or I.📌 holds final positions

In [ ]:
using GeoMakie, CairoMakie

fig = Figure(size=(900, 700))
ax  = GeoAxis(fig[1,1],
    dest     = "+proj=stere +lat_0=90 +lon_0=0",   # North Polar Stereographic
    title    = "90-day Arctic drifter tracks",
    lonlims  = (-180, 180),
    latlims  = (60, 90),
)

# Land and coastlines
GeoMakie.land!(ax, color=:lightgray)
GeoMakie.coastlines!(ax)

# Plot tracks (I.🌐 stores history if you set up a custom output function)
# Quick alternative: scatter final positions
lons_f = I.📌[1, :]
lats_f = I.📌[2, :]

scatter!(ax, lons_f, lats_f,
    color       = :orangered,
    markersize  = 6,
    transform_func = Makie.PointTrans{2}(p -> (p[1], p[2]))
)

# Mark seed points
scatter!(ax, x0, y0,
    color      = :royalblue,
    marker     = :diamond,
    markersize = 10,
    label      = "River mouths",
)

axislegend(ax)
save("arctic_drifters.png", fig)

In [ ]:
# Collect positions every 6 hours
Δt_out = 6 * 3600.0   # seconds

trajectory_log = DataFrame(id=Int[], t=Float64[], lon=Float64[], lat=Float64[])

function log_positions!(I, t)
    for (i, col) in enumerate(eachcol(I.📌))
        push!(trajectory_log, (i, t, col[1], col[2]))
    end
end

# Integrate with callback
∫!(I, (t_start, t_end),
    saveat   = t_start:Δt_out:t_end,
    callback = DiscreteCallback((u,t,integ)->true, integ->log_positions!(integ.p, integ.t))
)